In [3]:
import osmnx as ox

# Since U-Belt is informal, we query the main districts that comprise it
ubelt_districts = [
    "Sampaloc, Manila, Philippines", 
    "Quiapo, Manila, Philippines", 
    "San Miguel, Manila, Philippines"
]

try:
    # Fetch the geometries for these districts into a GeoDataFrame
    gdf = ox.geocode_to_gdf(ubelt_districts)
    
    # .total_bounds calculates the encompassing bounding box for all queried areas
    # Returns an array: [min_lon, min_lat, max_lon, max_lat]
    west, south, east, north = gdf.total_bounds
    
    print("Combined University Belt Bounding Box:")
    print(f"Min Longitude (West): {west}")
    print(f"Min Latitude (South): {south}")
    print(f"Max Longitude (East): {east}")
    print(f"Max Latitude (North): {north}")

except Exception as e:
    print(f"An error occurred: {e}")

Combined University Belt Bounding Box:
Min Longitude (West): 120.9807601
Min Latitude (South): 14.58955
Max Longitude (East): 121.0175128
Max Latitude (North): 14.6260417


In [1]:
import folium
import geopandas as gpd
from shapely.geometry import box as shapely_box

# ── Bounding box coordinates ──────────────────────────────────────────────────
SOUTH, NORTH = 14.5750, 14.6100
WEST,  EAST  = 120.9850, 121.0150

bbox_polygon = shapely_box(WEST, SOUTH, EAST, NORTH)
bbox_center  = [(SOUTH + NORTH) / 2, (WEST + EAST) / 2]

# ── Target universities (used to justify the bbox extent) ─────────────────────
universities_ref = [
    {"name": "University of Santo Tomas (UST)",            "lat": 14.6099, "lon": 120.9893},
    {"name": "Polytechnic Univ. of the Philippines (PUP)", "lat": 14.5981, "lon": 121.0092},
    {"name": "Pamantasan ng Lungsod ng Maynila (PLM)",     "lat": 14.5943, "lon": 120.9834},
    {"name": "De La Salle University (DLSU)",              "lat": 14.5648, "lon": 120.9932},
    {"name": "Mapua University",                           "lat": 14.5889, "lon": 120.9837},
    {"name": "Far Eastern University (FEU)",               "lat": 14.6010, "lon": 120.9893},
    {"name": "Centro Escolar University (CEU)",            "lat": 14.5977, "lon": 120.9913},
    {"name": "San Sebastian College",                      "lat": 14.5942, "lon": 120.9827},
]

# ── Check which universities fall inside the bbox ─────────────────────────────
print("University coverage check:")
print(f"{'University':<45} {'Lat':>8} {'Lon':>9} {'Inside bbox?':>14}")
print("-" * 80)
all_inside = True
for u in universities_ref:
    inside = (SOUTH <= u["lat"] <= NORTH) and (WEST <= u["lon"] <= EAST)
    if not inside:
        all_inside = False
    flag = "YES" if inside else "NO  ← OUTSIDE"
    print(f"{u['name']:<45} {u['lat']:>8.4f} {u['lon']:>9.4f} {flag:>14}")

print()
if all_inside:
    print("All target universities fall within the bounding box.")
else:
    print("WARNING: One or more universities fall outside the bounding box.")
    print("Consider adjusting the southern/northern/eastern/western boundary.")

# ── Bbox diagnostics ──────────────────────────────────────────────────────────
print(f"\nBounding box dimensions:")
print(f"  N-S span : {(NORTH - SOUTH) * 111:.2f} km  ({NORTH - SOUTH:.4f}°)")
print(f"  E-W span : {(EAST  - WEST)  * 111 * abs(__import__('math').cos(__import__('math').radians((SOUTH+NORTH)/2))):.2f} km  ({EAST - WEST:.4f}°)")

# ── Folium map ────────────────────────────────────────────────────────────────
m = folium.Map(location=bbox_center, zoom_start=13, tiles="CartoDB positron")

# Study area bbox
folium.Rectangle(
    bounds=[[SOUTH, WEST], [NORTH, EAST]],
    color="#185FA5",
    weight=2.5,
    fill=True,
    fill_color="#378ADD",
    fill_opacity=0.08,
    dash_array="8 4",
    tooltip="Study area bounding box"
).add_to(m)

# Boundary coordinate labels
corners = [
    (NORTH, WEST, f"NW ({NORTH}, {WEST})", "topleft"),
    (NORTH, EAST, f"NE ({NORTH}, {EAST})", "topright"),
    (SOUTH, WEST, f"SW ({SOUTH}, {WEST})", "bottomleft"),
    (SOUTH, EAST, f"SE ({SOUTH}, {EAST})", "bottomright"),
]
for lat, lon, label, anchor in corners:
    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(
            html=f'<div style="font-size:11px;font-weight:600;color:#185FA5;white-space:nowrap;">{label}</div>',
            icon_size=(180, 20),
        )
    ).add_to(m)

# Makati exclusion zone
folium.Rectangle(
    bounds=[[14.5500, 121.0150], [14.5850, 121.0500]],
    color="#E24B4A",
    weight=1.5,
    fill=True,
    fill_color="#E24B4A",
    fill_opacity=0.07,
    dash_array="5 4",
    tooltip="Makati — excluded (eastern boundary at 121.0150 cuts this off)"
).add_to(m)
folium.Marker(
    location=[14.5680, 121.0280],
    icon=folium.DivIcon(
        html='<div style="font-size:12px;font-weight:600;color:#A32D2D;">Makati (excluded)</div>',
        icon_size=(140, 20),
    )
).add_to(m)

# University markers
for u in universities_ref:
    inside = (SOUTH <= u["lat"] <= NORTH) and (WEST <= u["lon"] <= EAST)
    color  = "#1D9E75" if inside else "#D85A30"
    label  = u["name"] + (" ⚠ outside bbox" if not inside else "")
    folium.CircleMarker(
        location=[u["lat"], u["lon"]],
        radius=7,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.9,
        tooltip=label,
        popup=folium.Popup(
            f"<b>{u['name']}</b><br>Lat: {u['lat']}<br>Lon: {u['lon']}<br>{'✓ Inside bbox' if inside else '✗ Outside bbox'}",
            max_width=220
        )
    ).add_to(m)

# Legend
legend_html = """
<div style="position:fixed;bottom:30px;right:30px;z-index:9999;
     background:white;padding:12px 16px;border-radius:8px;
     border:1px solid #ccc;font-size:13px;line-height:2;">
  <b style="font-size:13px;">Legend</b><br>
  <span style="color:#185FA5;">&#9632;</span> Study bounding box<br>
  <span style="color:#1D9E75;">&#9679;</span> University inside bbox<br>
  <span style="color:#D85A30;">&#9679;</span> University outside bbox<br>
  <span style="color:#E24B4A;">&#9632;</span> Makati (excluded)
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m.save("outputs/bbox_justification_map.html")
print("\nMap saved to outputs/bbox_justification_map.html")
m

University coverage check:
University                                         Lat       Lon   Inside bbox?
--------------------------------------------------------------------------------
University of Santo Tomas (UST)                14.6099  120.9893            YES
Polytechnic Univ. of the Philippines (PUP)     14.5981  121.0092            YES
Pamantasan ng Lungsod ng Maynila (PLM)         14.5943  120.9834  NO  ← OUTSIDE
De La Salle University (DLSU)                  14.5648  120.9932  NO  ← OUTSIDE
Mapua University                               14.5889  120.9837  NO  ← OUTSIDE
Far Eastern University (FEU)                   14.6010  120.9893            YES
Centro Escolar University (CEU)                14.5977  120.9913            YES
San Sebastian College                          14.5942  120.9827  NO  ← OUTSIDE

Consider adjusting the southern/northern/eastern/western boundary.

Bounding box dimensions:
  N-S span : 3.89 km  (0.0350°)
  E-W span : 3.22 km  (0.0300°)

Map saved to 